# BinarySearch (C++)

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap13_correttezza/BinarySearch_cpp.ipynb)

Capitolo 13 — Correttezza. Notebook eseguibile di `BinarySearch`: l'algoritmo in C++, eseguito sugli esempi del capitolo, con la cella «Prova tu» per provarlo con dati tuoi. Lo pseudocodice, gli esempi e le dimostrazioni sono nella [pagina del capitolo](https://fabiofurini.github.io/algoritmi/analisi/13-correttezza/).

Lo stesso notebook in [Python](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap13_correttezza/BinarySearch_python.ipynb) e [Java](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap13_correttezza/BinarySearch_java.ipynb); tutti gli algoritmi del capitolo nel [notebook del capitolo](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap13_correttezza_cpp.ipynb).

**Come si esegue.** Clic sul badge «Apri in Colab», poi *Runtime → Esegui tutto* (Ctrl+F9), oppure una cella alla volta con Maiusc+Invio, dall'alto in basso. Ogni programma C++ è scritto su un file dalla cella `%%writefile`, poi compilato con `g++` ed eseguito nella cella successiva: tutto avviene in Colab.

**Indici.** Nello pseudocodice gli array sono indicizzati da 1; nel codice C++ da 0: l'elemento `a[i]` dello pseudocodice è `a[i-1]` nel codice (e un ciclo `while i <= n` da 1 diventa `while i < n` da 0). Le tabelle stampate usano gli indici dello pseudocodice.

## Ricerca binaria: BinarySearch

**Input:** un array $a[1..n]$ ordinato in modo non decrescente e un valore $m$. **Output:** un indice $k$ con $a[k] = m$ se $m$ è in $a$, $-1$ altrimenti. **Invariante:** (I1) $1 \le left$ e $right \le n$; (I2) se $m$ è in $a$, allora $m$ è in $a[left..right]$.

Pseudocodice: sezione 13.4 della [pagina del capitolo](https://fabiofurini.github.io/algoritmi/analisi/13-correttezza/).

In [1]:
%%writefile binary_search.cpp
#include <algorithm>
#include <cstdio>
#include <string>
using namespace std;

const int MAXN = 64;

// "[a[from], ..., a[to-1]]"
string Fmt(const int* a, int from, int to)
{
    string s = "[";
    for (int i = from; i < to; i++)
    {
        if (i > from) s += ", ";
        s += to_string(a[i]);
    }
    return s + "]";
}

bool Equal(const int* x, const int* y, int len)     // x[0..len-1] == y[0..len-1]
{
    for (int k = 0; k < len; k++)
        if (x[k] != y[k]) return false;
    return true;
}

// Invariante (I1) 1 <= left e right <= n, (I2) se m e' in a allora e' in a[left..right]; mid = -1: intervallo vuoto
void PrintRow(const int* a, int n, int m, int left, int right, int mid)
{
    bool inA = false, inSub = false;
    for (int k = 0; k < n; k++)
    {
        if (a[k] == m)
        {
            inA = true;
            if (k >= left && k <= right) inSub = true;
        }
    }
    bool ok = (left + 1 >= 1 && right + 1 <= n && (!inA || inSub));
    char action[100];
    if (mid < 0)
        snprintf(action, sizeof action, "left > right => return -1");
    else if (a[mid] == m)
        snprintf(action, sizeof action, "mid = %d, a[%d] = %d = %d => return %d", mid + 1, mid + 1, a[mid], m, mid + 1);
    else if (a[mid] < m)
        snprintf(action, sizeof action, "mid = %d, a[%d] = %d < %d => left = %d", mid + 1, mid + 1, a[mid], m, mid + 2);
    else
        snprintf(action, sizeof action, "mid = %d, a[%d] = %d > %d => right = %d", mid + 1, mid + 1, a[mid], m, mid);
    printf("%4d | %5d | %-27s | %-11s | %s\n", left + 1, right + 1, Fmt(a, left, right + 1).c_str(), ok ? "ok" : "FALSA", action);
}

// BinarySearch(a, n, m)
int BinarySearch(int a[], int n, int m, bool trace = false)
{
    int left = 0;                                  // left := 1
    int right = n - 1;                             // right := n
    while (left <= right)
    {
        int mid = left + (right - left) / 2;
        if (trace) PrintRow(a, n, m, left, right, mid);   // riga dell'invariante a questa valutazione
        if (a[mid] == m)
        {
            return mid;
        }
        if (a[mid] < m)
        {
            left = mid + 1;
        }
        else
        {
            right = mid - 1;
        }
    }
    if (trace) PrintRow(a, n, m, left, right, -1);
    return -1;
}

#ifndef SOLO_LIBRERIA
int main()
{
    int a[] = {3, 9, 11, 12, 12, 16, 19};          // esempio delle dispense
    int n = 7;
    printf("a = %s, n = %d\n", Fmt(a, 0, n).c_str(), n);

    int m = 10;
    printf("\n");
    printf("m = %d\n", m);
    printf("left | right | a[left..right]              | invariante  | azione\n");
    int k = BinarySearch(a, n, m, true);
    printf("m = %d non trovato: restituisce %d\n", m, k);

    m = 12;
    printf("\n");
    printf("m = %d\n", m);
    printf("left | right | a[left..right]              | invariante  | azione\n");
    k = BinarySearch(a, n, m, true);
    printf("m = %d trovato: indice %d nel codice (posizione %d nello pseudocodice)\n", m, k, k + 1);

    // Controllo esaustivo per m = 0, ..., 20
    int errors = 0;
    for (m = 0; m <= 20; m++)
    {
        k = BinarySearch(a, n, m);
        bool inA = false;
        for (int t = 0; t < n; t++) if (a[t] == m) inA = true;
        if ((inA && (k < 0 || a[k] != m)) || (!inA && k != -1)) errors++;
    }
    printf("\n");
    printf("Controllo per m = 0, ..., 20: %d errori\n", errors);
    return 0;
}
#endif

Writing binary_search.cpp


In [2]:
!g++ -std=c++17 -O2 -o binary_search binary_search.cpp && ./binary_search

a = [3, 9, 11, 12, 12, 16, 19], n = 7

m = 10
left | right | a[left..right]              | invariante  | azione
   1 |     7 | [3, 9, 11, 12, 12, 16, 19]  | ok          | mid = 4, a[4] = 12 > 10 => right = 3
   1 |     3 | [3, 9, 11]                  | ok          | mid = 2, a[2] = 9 < 10 => left = 3
   3 |     3 | [11]                        | ok          | mid = 3, a[3] = 11 > 10 => right = 2
   3 |     2 | []                          | ok          | left > right => return -1
m = 10 non trovato: restituisce -1

m = 12
left | right | a[left..right]              | invariante  | azione
   1 |     7 | [3, 9, 11, 12, 12, 16, 19]  | ok          | mid = 4, a[4] = 12 = 12 => return 4
m = 12 trovato: indice 3 nel codice (posizione 4 nello pseudocodice)

Controllo per m = 0, ..., 20: 0 errori


**Prova tu.** Cambia l'array (deve restare ordinato) e il valore cercato.

In [3]:
%%writefile prova_binary_search.cpp
#define SOLO_LIBRERIA   // non eseguire il main di binary_search.cpp
#include "binary_search.cpp"

int main()
{
    int a[] = {2, 4, 4, 7, 9, 15, 21, 30};      // <-- cambia l'array (ordinato!)
    int n = 8;
    int m = 15;                                 // <-- e il valore cercato
    printf("left | right | a[left..right]              | invariante  | azione\n");
    int k = BinarySearch(a, n, m, true);
    printf("restituisce %d\n", k);
    return 0;
}

Writing prova_binary_search.cpp


In [4]:
!g++ -std=c++17 -O2 -o prova_binary_search prova_binary_search.cpp && ./prova_binary_search

left | right | a[left..right]              | invariante  | azione
   1 |     8 | [2, 4, 4, 7, 9, 15, 21, 30] | ok          | mid = 4, a[4] = 7 < 15 => left = 5
   5 |     8 | [9, 15, 21, 30]             | ok          | mid = 6, a[6] = 15 = 15 => return 6
restituisce 5


---

Materiale didattico di [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza Università di Roma.
Testo, figure e dati [CC BY 4.0](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE), codice [MIT](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE-CODE).